# Ingenieria de caracteristicas: Target Encoding (Bayesiano) y WOE Encoding

Este notebook es una variante de [`03_feature_engineering.ipynb`](./03_feature_engineering.ipynb) enfocada unicamente en la ingenieria de caracteristicas categoricas del modelo de riesgo de paralizacion de obras publicas, usando una estrategia de codificacion distinta:

1. **Target encoding con suavizado bayesiano** para variables categoricas de cardinalidad media/alta: `tipo_obra_nivel_2`, `tipo_obra_nivel_3`, `departamento`, `provincia`.
2. **WOE (Weight of Evidence) encoding con suavizado** para `codigo_entidad` y `sector_entidad`.
3. **One-hot encoding** para variables categoricas de baja cardinalidad: `naturaleza_obra`, `tipo_formato`.
4. Se mantiene la **particion temporal** (train/test por `fecha_inicio_obra`) usada en la seccion `## Aplicacion de modelo de clasificacion` de `03_feature_engineering.ipynb`, y ambos encoders supervisados se ajustan **solo sobre train** para evitar fuga de datos.

Los transformadores (`BayesianTargetEncoder`, `WOEEncoder`) y los constructores de pipelines/preprocesador viven en [`src/features/encoders.py`](../src/features/encoders.py) y [`src/features/preprocessing.py`](../src/features/preprocessing.py) para poder reutilizarse desde otros notebooks y scripts.

## 1. Configuracion inicial

In [1]:
import sys
from configparser import ConfigParser
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Agregar 'src' al path para importar los modulos del proyecto
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.db.connection import PostgresConnection
from src.features.preprocessing import build_preprocessor
from src.experiments.tracking import create_experiment_run

print(f"Fecha de ejecucion: {datetime.now():%Y-%m-%d %H:%M:%S}")


Fecha de ejecucion: 2026-09-04 10:53:45


### Conexion a la fuente de datos y carga

In [2]:
config_path = project_root / "configs" / "database.ini"
config = ConfigParser()
config.read(config_path)

pg_conn = PostgresConnection(
    host=config.get("postgres", "host"),
    port=config.getint("postgres", "port"),
    dbname=config.get("postgres", "dbname"),
    user=config.get("postgres", "user"),
    password=config.get("postgres", "password"),
)
conn = pg_conn.connect()
print(f"Conexion establecida: {pg_conn.host}:{pg_conn.port}/{pg_conn.dbname}")


Conexion establecida: localhost:5432/postgres


In [3]:
# Misma tabla y filtro funcional que 03_feature_engineering.ipynb:
# nivel de gobierno nacional, marco INVIERTE.
TABLA_ORIGEN = "r$_ds_obras_publicas_infobras"
NIVEL_GOBIERNO_FILTRO = "GOBIERNO NACIONAL"
MARCO = "INVIERTE"

df_raw = pd.read_sql(f"SELECT * FROM {TABLA_ORIGEN}", conn)
df = df_raw[
    (df_raw["nivel_gobierno"] == NIVEL_GOBIERNO_FILTRO) & (df_raw["marco"] == MARCO)
].copy()

print(f"Registros cargados: {len(df_raw):,}")
print(f"Registros tras el filtro: {len(df):,}")


C:\Users\yanfr\AppData\Local\Temp\ipykernel_27748\1445047479.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_raw = pd.read_sql(f"SELECT * FROM {TABLA_ORIGEN}", conn)


Registros cargados: 69,952
Registros tras el filtro: 2,646


## 2. Seleccion de columnas

Se seleccionan unicamente las columnas necesarias para este pipeline: el target, la fecha usada para la particion temporal, el monto (para la limpieza de outliers descrita en `03_feature_engineering.ipynb`), y las 8 variables categoricas indicadas en el punto 1-3 de la consigna. No se incluyen las tasas historicas precalculadas en la fuente (`tasa_paralizacion_*`, `monto_aprobado_soles_promedio_entidad`, `n_obras_x_entidad`) porque este notebook las reemplaza por los nuevos encoders, ajustados correctamente solo sobre train.

In [4]:
columnas_target_encoding = ["tipo_obra_nivel_2", "tipo_obra_nivel_3", "departamento", "provincia"]
columnas_woe = ["codigo_entidad", "sector_entidad"]
columnas_onehot = ["naturaleza_obra", "tipo_formato"]
columna_monto = "monto_aprobado_soles"
columna_monto_referencia = "monto_aprobacion_et"  # usada solo para corregir monto_aprobado_soles == 0
columna_numerica = "n_obras_relacionadas_proyecto"
DATE_COL = "fecha_inicio_obra"
TARGET = "flg_opa_incumplimiento_contractual"

columnas_interes = (
    columnas_target_encoding
    + columnas_woe
    + columnas_onehot
    + [columna_monto, columna_monto_referencia, columna_numerica, DATE_COL, TARGET]
)

df_reducido = df[columnas_interes].copy()
df_reducido.info()


<class 'pandas.DataFrame'>
Index: 2646 entries, 68 to 69948
Data columns (total 13 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   tipo_obra_nivel_2                   2646 non-null   str    
 1   tipo_obra_nivel_3                   2646 non-null   str    
 2   departamento                        2645 non-null   str    
 3   provincia                           2645 non-null   str    
 4   codigo_entidad                      2646 non-null   str    
 5   sector_entidad                      2646 non-null   str    
 6   naturaleza_obra                     2646 non-null   str    
 7   tipo_formato                        2551 non-null   str    
 8   monto_aprobado_soles                2628 non-null   float64
 9   monto_aprobacion_et                 2628 non-null   float64
 10  n_obras_relacionadas_proyecto       2646 non-null   int64  
 11  fecha_inicio_obra                   2646 non-null   objec

## 3. Limpieza de `monto_aprobado_soles`

Se replica el tratamiento de `03_feature_engineering.ipynb` (seccion `4.3 Transformaciones para outliers`): eliminar nulos, reemplazar los montos en 0 por `monto_aprobacion_et`, y descartar el registro con `monto_aprobado_soles == 1`. La winsorizacion y el logaritmo del monto se calculan **dentro** del pipeline de sklearn (`WinsorLogTransformer`), ajustado solo sobre train, para no filtrar informacion del test.

In [5]:
n_antes = len(df_reducido)
df_reducido = df_reducido.dropna(subset=[columna_monto])
print(f"Registros eliminados por {columna_monto} nulo: {n_antes - len(df_reducido):,}")


Registros eliminados por monto_aprobado_soles nulo: 18


In [6]:
# monto_aprobado_soles == 0 -> se reemplaza por monto_aprobacion_et
mask_monto_cero = df_reducido[columna_monto] == 0
df_reducido.loc[mask_monto_cero, columna_monto] = df_reducido.loc[mask_monto_cero, columna_monto_referencia]
print(f"Registros corregidos (monto == 0): {mask_monto_cero.sum():,}")

# monto_aprobado_soles == 1 -> registro invalido, se descarta
n_antes = len(df_reducido)
df_reducido = df_reducido[df_reducido[columna_monto] != 1].copy()
print(f"Registros eliminados (monto == 1): {n_antes - len(df_reducido):,}")


Registros corregidos (monto == 0): 22
Registros eliminados (monto == 1): 1


## 4. Particion temporal (train / test)

Se mantiene el mismo criterio que la seccion `## Aplicacion de modelo de clasificacion` (`M2`) de `03_feature_engineering.ipynb`: se ordena por `fecha_inicio_obra` y se separa el 65% mas antiguo para train y el 35% mas reciente para test, respetando la naturaleza temporal del fenomeno (se usa la historia de ejecuciones para estimar el riesgo de obras futuras).

In [7]:
df_reducido[DATE_COL] = pd.to_datetime(df_reducido[DATE_COL], errors="coerce")
df_reducido = df_reducido.dropna(subset=[DATE_COL, TARGET])
df_reducido = df_reducido.sort_values(DATE_COL).reset_index(drop=True)

split_idx = int(len(df_reducido) * 0.65)
df_train = df_reducido.iloc[:split_idx].copy()
df_test = df_reducido.iloc[split_idx:].copy()

print(f"Train: {len(df_train):,} registros | {df_train[DATE_COL].min().date()} a {df_train[DATE_COL].max().date()}")
print(f"Test : {len(df_test):,} registros | {df_test[DATE_COL].min().date()} a {df_test[DATE_COL].max().date()}")


Train: 1,707 registros | 2010-12-18 a 2023-01-11
Test : 920 registros | 2023-01-11 a 2025-12-20


In [8]:
display(Markdown("**Tasa de paralizacion - train**"))
display(Markdown(df_train[TARGET].value_counts(normalize=True).to_markdown()))

display(Markdown("**Tasa de paralizacion - test**"))
display(Markdown(df_test[TARGET].value_counts(normalize=True).to_markdown()))


**Tasa de paralizacion - train**

|   flg_opa_incumplimiento_contractual |   proportion |
|-------------------------------------:|-------------:|
|                                    0 |    0.915641  |
|                                    1 |    0.0843585 |

**Tasa de paralizacion - test**

|   flg_opa_incumplimiento_contractual |   proportion |
|-------------------------------------:|-------------:|
|                                    0 |    0.942391  |
|                                    1 |    0.0576087 |

In [9]:
columnas_features = (
    columnas_target_encoding + columnas_woe + columnas_onehot
    + [columna_monto, columna_numerica]
)

X_train = df_train[columnas_features]
y_train = df_train[TARGET]
X_test = df_test[columnas_features]
y_test = df_test[TARGET]


## 5. Preprocesador: target encoding + WOE + one-hot

Se construye un unico `ColumnTransformer` con `build_preprocessor` (`src/features/preprocessing.py`), que enruta cada grupo de columnas a su encoder correspondiente:

| Columnas | Metodo | Suavizado |
|---|---|---|
| `tipo_obra_nivel_2`, `tipo_obra_nivel_3`, `departamento`, `provincia` | Target encoding bayesiano | `smoothing=20` |
| `codigo_entidad`, `sector_entidad` | WOE | `smoothing=20`, `eps=0.5` |
| `naturaleza_obra`, `tipo_formato` | One-hot | `min_frequency=20` |
| `monto_aprobado_soles` | Winsorizacion (p99) + log1p + estandarizacion | - |
| `n_obras_relacionadas_proyecto` | Estandarizacion | - |

`target_encoded_columns` y `woe_columns` son transformadores supervisados: se ajustan usando el `y` que reciba el `ColumnTransformer.fit`, por lo que **solo se llama `fit`/`fit_transform` con los datos de train**; el test se transforma reutilizando los mapeos ya ajustados (`transform`), igual que el resto del pipeline.

In [10]:
preprocessor = build_preprocessor(
    amount_log_columns=[columna_monto],
    numeric_columns=[columna_numerica],
    categorical_columns=columnas_onehot,
    target_encoded_columns=columnas_target_encoding,
    woe_columns=columnas_woe,
    target_encoding_smoothing=20.0,
    woe_smoothing=20.0,
    woe_eps=0.5,
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f"X_train_processed: {X_train_processed.shape}")
print(f"X_test_processed : {X_test_processed.shape}")
print(f"Features generadas: {len(feature_names)}")
list(feature_names)


X_train_processed: (1707, 21)
X_test_processed : (920, 21)
Features generadas: 21


['amount_log__monto_aprobado_soles_winsor_log',
 'numeric_standard__n_obras_relacionadas_proyecto',
 'categorical__naturaleza_obra_Ampliación',
 'categorical__naturaleza_obra_Construcción/Creación',
 'categorical__naturaleza_obra_Instalación',
 'categorical__naturaleza_obra_Mejoramiento',
 'categorical__naturaleza_obra_Reconstrucción',
 'categorical__naturaleza_obra_Rehabilitación',
 'categorical__naturaleza_obra_Remodelación',
 'categorical__naturaleza_obra_Renovación',
 'categorical__naturaleza_obra_Reparación',
 'categorical__naturaleza_obra_infrequent_sklearn',
 'categorical__tipo_formato_1',
 'categorical__tipo_formato_2',
 'categorical__tipo_formato_5',
 'target_encoded__tipo_obra_nivel_2_target_enc',
 'target_encoded__tipo_obra_nivel_3_target_enc',
 'target_encoded__departamento_target_enc',
 'target_encoded__provincia_target_enc',
 'woe__codigo_entidad_woe',
 'woe__sector_entidad_woe']

## 6. Diagnostico de los encoders

Se inspecciona lo que aprendio cada encoder sobre train: los valores de target encoding para las categorias mas frecuentes, y el **Information Value (IV)** de cada variable codificada con WOE (regla de pulgar: <0.02 no predictiva, 0.02-0.1 debil, 0.1-0.3 media, 0.3-0.5 fuerte, >0.5 sospechosamente fuerte / posible fuga de datos).

In [11]:
target_encoder_step = preprocessor.named_transformers_["target_encoded"].named_steps["target_encoder"]

for col in columnas_target_encoding:
    mapping = target_encoder_step.mappings_[col]
    top_categorias = sorted(mapping.items(), key=lambda kv: kv[1], reverse=True)[:5]
    display(Markdown(f"**{col}** (tasa global = {target_encoder_step.global_mean_:.4f}) - top 5 por tasa codificada"))
    display(pd.DataFrame(top_categorias, columns=[col, "tasa_codificada"]))


**tipo_obra_nivel_2** (tasa global = 0.0844) - top 5 por tasa codificada

,tipo_obra_nivel_2,tasa_codificada
0,Riego,0.230731
1,Transporte Terrestre,0.137189
2,Desarrollo Urbano Y Rural,0.136562
3,Transporte Aéreo,0.107487
4,Energía Eléctrica,0.105676


**tipo_obra_nivel_3** (tasa global = 0.0844) - top 5 por tasa codificada

,tipo_obra_nivel_3,tasa_codificada
0,Canales,0.216068
1,Almacenamiento Y Distribución,0.203790
2,Sistema De Distribución Y Transmisión Eléctrica,0.192179
3,Generación / Distribución Y Transmisión Eléctrica,0.156239
4,Puentes,0.140363


**departamento** (tasa global = 0.0844) - top 5 por tasa codificada

,departamento,tasa_codificada
0,TUMBES,0.187494
1,ANCASH,0.133964
2,JUNIN,0.131624
3,CAJAMARCA,0.106766
4,P C DEL CALLAO,0.095970


**provincia** (tasa global = 0.0844) - top 5 por tasa codificada

,provincia,tasa_codificada
0,PALLASCA,0.230592
1,ASUNCION,0.195299
2,JAEN,0.191062
3,ZARUMILLA,0.173599
4,TUMBES,0.170826


In [12]:
woe_encoder_step = preprocessor.named_transformers_["woe"].named_steps["woe_encoder"]

iv_resumen = pd.Series(woe_encoder_step.iv_, name="information_value").sort_values(ascending=False)
display(Markdown("**Information Value por variable (WOE)**"))
display(iv_resumen.to_frame())


**Information Value por variable (WOE)**

,information_value
codigo_entidad,1.226565
sector_entidad,0.560053


In [13]:
for col in columnas_woe:
    mapping = woe_encoder_step.mappings_[col]
    top_categorias = sorted(mapping.items(), key=lambda kv: kv[1], reverse=True)[:5]
    display(Markdown(f"**{col}** - categorias con WOE mas alto (mayor riesgo relativo)"))
    display(pd.DataFrame(top_categorias, columns=[col, "woe"]))


**codigo_entidad** - categorias con WOE mas alto (mayor riesgo relativo)

,codigo_entidad,woe
0,4047,0.781137
1,164,0.772935
2,4812,0.723235
3,5297,0.687053
4,5741,0.656627


**sector_entidad** - categorias con WOE mas alto (mayor riesgo relativo)

,sector_entidad,woe
0,AGRICULTURA,0.887415
1,SALUD,0.565197
2,TRANSPORTES Y COMUNICACIONES,0.488254
3,CULTURA,0.182851
4,FONAFE,0.179756


## 7. Verificacion de ausencia de fuga de datos

Como el preprocesador se ajusto unicamente con `(X_train, y_train)`, los valores codificados en test para una categoria vista en train deben coincidir exactamente con el valor aprendido en train; las categorias nuevas en test deben caer en el valor neutro de cada encoder (media global para target encoding, 0 para WOE).

In [14]:
def verificar_sin_fuga(encoder_step, columnas, valor_neutro_fn, X_train, X_test, nombre_encoder):
    for col in columnas:
        mapping = encoder_step.mappings_[col]
        categorias_train = set(X_train[col].astype("object").fillna("__missing__"))
        categorias_test = set(X_test[col].astype("object").fillna("__missing__"))
        categorias_nuevas = categorias_test - categorias_train

        valores_train = {c: mapping[c] for c in categorias_train if c in mapping}
        coincide = all(mapping.get(c) == v for c, v in valores_train.items())

        print(
            f"[{nombre_encoder}] {col}: {len(categorias_train)} categorias en train, "
            f"{len(categorias_nuevas)} categorias nuevas en test "
            f"(valor neutro={valor_neutro_fn(col):.4f}), mapeo train==train: {coincide}"
        )


verificar_sin_fuga(
    target_encoder_step, columnas_target_encoding,
    lambda col: target_encoder_step.global_mean_,
    X_train, X_test, "target_encoding"
)
verificar_sin_fuga(
    woe_encoder_step, columnas_woe,
    lambda col: 0.0,
    X_train, X_test, "woe"
)


[target_encoding] tipo_obra_nivel_2: 30 categorias en train, 6 categorias nuevas en test (valor neutro=0.0844), mapeo train==train: True
[target_encoding] tipo_obra_nivel_3: 54 categorias en train, 12 categorias nuevas en test (valor neutro=0.0844), mapeo train==train: True
[target_encoding] departamento: 25 categorias en train, 1 categorias nuevas en test (valor neutro=0.0844), mapeo train==train: True
[target_encoding] provincia: 176 categorias en train, 13 categorias nuevas en test (valor neutro=0.0844), mapeo train==train: True
[woe] codigo_entidad: 126 categorias en train, 39 categorias nuevas en test (valor neutro=0.0000), mapeo train==train: True
[woe] sector_entidad: 20 categorias en train, 3 categorias nuevas en test (valor neutro=0.0000), mapeo train==train: True


## 8. Persistir artefactos

Se guarda la corrida usando `create_experiment_run` (`src/experiments/tracking.py`): configuracion de encoders, matrices procesadas de train/test, targets, nombres de features, Information Value de las variables WOE y el preprocesador ajustado (`preprocessor.joblib`), bajo `artifacts/experiments/`.

In [15]:
import joblib
from scipy import sparse

run = create_experiment_run(name="feature_engineering_target_woe_encoding")

run.save_config({
    "date_col": DATE_COL,
    "target": TARGET,
    "train_test_split_ratio": 0.65,
    "columnas_target_encoding": columnas_target_encoding,
    "columnas_woe": columnas_woe,
    "columnas_onehot": columnas_onehot,
    "target_encoding_smoothing": 20.0,
    "woe_smoothing": 20.0,
    "woe_eps": 0.5,
})

run.save_json({"information_value": woe_encoder_step.iv_}, "woe_information_value.json")

X_train_dense = X_train_processed.toarray() if sparse.issparse(X_train_processed) else X_train_processed
X_test_dense = X_test_processed.toarray() if sparse.issparse(X_test_processed) else X_test_processed

run.save_dataframe_csv(pd.DataFrame(X_train_dense, columns=feature_names), "X_train_processed.csv")
run.save_dataframe_csv(pd.DataFrame(X_test_dense, columns=feature_names), "X_test_processed.csv")
run.save_dataframe_csv(y_train.to_frame(), "y_train.csv")
run.save_dataframe_csv(y_test.to_frame(), "y_test.csv")
run.save_dataframe_csv(pd.Series(feature_names, name="feature").to_frame(), "feature_names.csv")

joblib.dump(preprocessor, run.path / "preprocessor.joblib")

print(f"Artefactos guardados en: {run.path}")


Artefactos guardados en: artifacts\experiments\2026-09-04_110033_feature_engineering_target_woe_encoding


## 9. Cerrar conexion a la BD

In [ ]:
if pg_conn:
    pg_conn.close()
    print("Conexion a PostgreSQL cerrada")
